## Status Check

In [3]:
!nvidia-smi

Thu Aug 13 20:58:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.153.02             Driver Version: 570.153.02     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               On  |   00000000:A1:00.0 Off |                  Off |
| 30%   40C    P8             35W /  300W |       1MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## SYSTEM VAR

In [6]:
DEFAULT_MAX_TOKENS = 512
QWEN3=True
SEED=42

## Library Import

In [7]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, Qwen2_5_VLForConditionalGeneration
import torch

from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.build_sam import build_sam2

# from sam3.model_builder import build_sam3_image_model
# from sam3.model.sam3_image_processor import Sam3Processor
# from sam3.model.box_ops import box_xywh_to_cxcywh
# from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import json
import matplotlib.patches as patches
from dataclasses import dataclass, field
from collections import Counter

torch.cuda.is_available()

True

## Default Function Init

In [8]:
def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed) 
    random.seed(seed)  
    np.random.seed(seed)  
    torch.manual_seed(seed)  
    torch.cuda.manual_seed(seed)  
    torch.cuda.manual_seed_all(seed)  
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

In [9]:
def encode_single_mask(mask):
    """
    Input:
        mask: [1000, 1000] 또는 [B, 1000, 1000] 형태의 bool 타입의 np.ndarray 혹은 torch.Tensor
    Returns:
        RLE 문자열
    """
    if torch.is_tensor(mask):
        mask = mask.detach().cpu().numpy()
    # 픽셀을 일렬로 세움 (Flatten ; order='F')
    pixels = mask.T.flatten()

    pixels = np.concatenate([[False], pixels, [False]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]

    # 결과 문자열 생성 (예: "1 3 10 5")
    return ' '.join(str(x) for x in runs)


## Model Load

In [10]:
MODEL_PATH = "/home/jjs2403/2026_bootcamp_02/models/Qwen3-VL-8B-Instruct"
qwen3_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_PATH, 
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto", 
    attn_implementation="flash_attention_2"
)
qwen3_processor = AutoProcessor.from_pretrained(MODEL_PATH)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [11]:
# MODEL_PATH = "/home/jjs2403/2026_bootcamp_02/models/Qwen2.5-VL-7B-Instruct"

# qwen25_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_PATH,
#     dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto",
#     attn_implementation="flash_attention_2",
# )

# qwen25_processor = AutoProcessor.from_pretrained(MODEL_PATH)


In [12]:
SAM2_1_MODEL_PATH = "//home//jjs2403//2026_bootcamp_02//models//SAM2.1//weights//sam2.1_hiera_large.pt"
SAM2_1_MODEL_CFG_PATH = "//home//jjs2403//2026_bootcamp_02//models//SAM2.1//weights//sam2.1_hiera_l.yaml"

sam21_mask_predictor = SAM2ImagePredictor(build_sam2(SAM2_1_MODEL_CFG_PATH, SAM2_1_MODEL_PATH))

In [13]:
# SAM3_MODEL_PATH = "/home/jjs2403/2026_bootcamp_summer/models/sam3/sam3.pt"
# device = 'cuda:0'
# sam3_model = build_sam3_image_model(
#     eval_mode=True,
#     checkpoint_path="/home/jjs2403/2026_bootcamp_summer/models/sam3/sam3.pt",
#     load_from_HF=False,
#     enable_segmentation=True,
#     enable_inst_interactivity=False,
#     compile=False,
# ).to(device)
# sam3_processor = Sam3Processor(sam3_model)


## Dataset Load

In [14]:
class Task2Dataset(Dataset):
    """
    Dataset 클래스.

    Return 값은 다음과 같다.
    image_id: image번호(000__.png)
    image: PIL 원본이미지
    query: 해당 이미지의 쿼리
    """
    def __init__(self, data_path, query_path):
        self.data_path = data_path
        self.query_df = pd.read_csv(query_path)

    def __len__(self):
        return len(self.query_df)

    def __getitem__(self, idx):
        row = self.query_df.iloc[idx]
        image_id = int(row["ID"])
        query = row["Query"]

        image_name = f"{image_id:05d}.png" # 00001.png
        image_path = os.path.join(self.data_path, image_name)

        image = Image.open(image_path).convert("RGB")

        return image_id, image, query # image는 그냥 PIL 이미지가 반환되니까 주의!!!

In [15]:
BASE_PATH = "/home/jjs2403/2026_bootcamp_summer/_ASSIGNMENTS/competitions/competition2"

test_dataset = Task2Dataset(
    data_path="/home/jjs2403/2026_bootcamp_summer/_ASSIGNMENTS/competitions/competition2/imgs",
    query_path="/home/jjs2403/2026_bootcamp_summer/_ASSIGNMENTS/competitions/competition2/query.csv"
)

In [16]:
def collate_fn(batch):
    return batch[0]


test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)

## Template&Parsing Function

In [17]:
### @deprecated @fallback
### sub5 version(OLD)
def template_old(query): ##### DEFAULT TEMPLATE ####
    return f"""
Identify the target that answers the query or is referred to by the query in the image.

If the target can be identified directly, locate it directly.
If the query requires reasoning, counting, comparison, or solving, use the image information to determine the answer first and then locate the target.
Use only the reasoning needed for the query.

After determining the answer, locate that exact visual target in the image.
Before returning the result, verify that each bbox and point are on the object described by its concept, not on a nearby or related object.

Return only valid JSON in this format:
{{
    "description": "short description of the target",
    "targets": [
        {{
            "concept": "short visual noun phrase",
            "bbox_2d": [x1, y1, x2, y2],
            "point_2d": [x, y]
        }}
    ]
}}

The point must be inside the target.
If there are multiple targets, return all of them in the targets list.

Query: {query}
"""

In [18]:
def template_classify(query):
    return f"""
Classify the following query into exactly one reasoning type below, based on what kind of reasoning process is needed to find the target it refers to.

A. Direct Grounding - the query names or describes the target explicitly, with no reasoning needed.
B. Indirect Clue / Figurative - the target is described indirectly, through metaphor, behavior, or implied visual clues rather than named directly.
C. Functional & Situational Commonsense - the query describes a situation, need, or condition, and the target must be found using common-sense knowledge of what object serves that function.
D. Comparative / Ranking - the query requires comparing multiple candidate objects on some attribute and selecting one based on an extreme value or rank (e.g., most, least, second-highest).
E. Quantitative / Logical Reasoning - the query requires an explicit calculation or step-by-step logical deduction, such as arithmetic, date/calendar computation, solving a logic puzzle (e.g., Sudoku), or reading a value from a chart or graph.
F. Factual / Domain Knowledge - the query requires recalling a specific fact (e.g., game rules, regulations, categorical facts) to determine the correct target, without needing multi-step calculation or deduction.

Return only valid JSON in this format:
{{
    "reason": "brief explanation of why this type was chosen",
    "type": "A"
}}

Query: {query}
"""

In [19]:
def parse_query_type(output):
    """
    output 출력결과를 넣으면 qtype(A~F)과 reason을 반환하는 함수.
    단. Fallback시 None과 빈 String을 반환함.
    """
    output = output.replace("```json", "").replace("```", "").strip()

    try:
        start = output.find("{")
        end = output.rfind("}") + 1

        if start == -1 or end <= start:
            return None, ""

        result = json.loads(output[start:end])

        qtype = str(result.get("type", "")).strip().upper()
        reason = str(result.get("reason", "")).strip()

        if qtype not in ["A", "B", "C", "D", "E", "F"]:
            return None, reason

        return qtype, reason

    except:
        print("[ERROR] parse_query_type()")
        return None, ""

In [20]:
TYPE_INSTRUCTIONS = {
    "A": (
        "Identify the exact target described by the query in the image and locate it directly. "
        "Do not add unnecessary reasoning steps because the query already names or describes the target explicitly."
    ),

    "B": (
        "The query refers to the target indirectly, through metaphor, behavior, or an implied clue rather than naming it directly. "
        "First interpret what real-world object, person, or state the description is actually pointing to, "
        "then locate that exact target in the image."
    ),

    "C": (
        "The query describes a situation, need, or condition. "
        "First identify exactly what condition is being described, then find the object in the image that functionally satisfies it, "
        "based on common-sense knowledge of how such objects are normally used in that situation. "
        "Prefer the ordinary and commonly expected use or behavior. "
        "Do not rely on rare, unusual, or merely technically possible edge cases."
    ),

    "D": (
        "The query requires comparing multiple candidate objects in the image along some attribute. "
        "First identify all relevant candidates, evaluate the attribute for each one, "
        "then select the one matching the specified rank or extreme (e.g., most, least, second-highest). "
        "Double-check the rank before finalizing and do not confuse an extreme value with a specific rank."
    ),

    "E": (
        "The query requires a calculation, counting, numerical or time-based filtering, or logical deduction. "
        "Evaluate the entire condition rather than only one local step. "
        "Perform the required calculation or deduction first, then independently check the result once. "
        "If the verification contradicts the first result, recalculate before locating the corresponding target in the image."
    ),

    "F": (
        "The query requires specific factual or domain knowledge (e.g., game rules, official regulations, categorical facts) "
        "to determine the correct target. "
        "Recall the relevant fact first, then apply that fact consistently to identify and locate the correct target in the image."
    ),
}


def template_v2(query, qtype):
    if qtype is None:
        return template_old(query)

    instruction = TYPE_INSTRUCTIONS[qtype]

    return f"""
{instruction}

Before solving the query, identify its exact segmentation requirements:
- the requested number or quantifier,
- every explicitly requested visual component,
- and every exclusion or negative condition.

Preserve all of these requirements exactly in the final targets list.
Do not omit one component when the query requests multiple components.
Do not include related or contextual objects that are not requested.

If the query specifies an exclusion (e.g., "excluding X", "not Y", "other than Z"), make sure the final target does not fall under the excluded case.

Before returning the result, verify that each bbox and point are on the object described by its concept, not on a nearby or related object.

The semantic type of each target must exactly match what the query asks to segment.

If multiple spatially separate targets satisfy the query, return a separate target entry for each one.
Do not use one large bounding box to cover multiple separate targets.

Assume that at least one valid target exists in the image.
Do not return an empty "targets" list.
If your first interpretation finds no valid target, reconsider the query and image and select the best matching visible target.

Keep the reasoning concise and non-repetitive.
Use at most 4 short sentences.
Do not enumerate every candidate, cell, or item individually when a shorter summary is sufficient.
Always leave enough room to complete the "description" and "targets" fields.

Return only valid JSON in this format:
{{
    "reasoning": "concise reasoning process used to determine the target",
    "description": "short description of the target",
    "targets": [
        {{
            "concept": "short visual noun phrase",
            "bbox_2d": [x1, y1, x2, y2],
            "point_2d": [x, y]
        }}
    ]
}}

The point must be inside the target.

Query: {query}
"""

In [21]:
def parse_output(output):
    """
    Qwen Output을 Json으로 Parsing하는 Funcion
    targets 리스트에 모든 BOX 저장
    """
    output = output.replace("```json", "").replace("```", "").strip()

    start = output.find("{")
    end = output.rfind("}") + 1

    result = json.loads(output[start:end])

    description = result.get("description", "")
    targets = []

    for item in result["targets"]: ## ALL TARGET SAVED
        targets.append({
            "concept": item["concept"].strip(),
            "bbox": item["bbox_2d"],
            "point": item["point_2d"]
        })

    return description, targets

def parse_output_v2(output):
    """
    reasoning 필드가 포함된 2단계 출력을 파싱하는 함수.
    """
    output = output.replace("```json", "").replace("```", "").strip()

    start = output.find("{")
    end = output.rfind("}") + 1

    result = json.loads(output[start:end])

    reasoning = result.get("reasoning", "")
    description = result.get("description", "")
    targets = []

    for item in result["targets"]:
        targets.append({
            "concept": item["concept"].strip(),
            "bbox": item["bbox_2d"],
            "point": item["point_2d"]
        })

    return description, targets, reasoning

## Inference Function

In [22]:
def qwen_get_response(model, processor, image, query, max_new_tokens=DEFAULT_MAX_TOKENS):
    model.eval()

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": query},
            ],
        }
    ]

    input_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[input_text],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    return processor.batch_decode(
        output_ids[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )[0]

In [23]:
def qwen_inference_v2(model, processor, image, query, max_retry=1): ## MAX RETRY는 웬만해선 1로 고정할것. 2해봤자 sampling 없어서 똑같음 ㅇㅇ
    """
    RETURN값
    output, description, targets, qtype, cls_reason, reasoning, status(FALLBACK or not)
    """
    ############# STAGE 1 ##################
    qtype = None # Question Type
    cls_reason = "" # Classfication Reason.

    for retry in range(max_retry):
        cls_output = qwen_get_response(model, processor,image,template_classify(query))

        qtype, cls_reason = parse_query_type(cls_output)

        if qtype is not None:
            break

        print(f"[WARN] Query Type Parsing Error - Retry {retry + 1}/{max_retry}")

    # Classfication fail
    if qtype is None:
        print("Query Type Classification Failed - Use Default Prompt")

        output = qwen_get_response(model, processor,image,template_old(query)) ## FALLBACK ##

        try:
            description, targets = parse_output(output)
            return output, description, targets, None, cls_reason, "", "classifier_fallback_success"
        except:
            print("[ERROR] A: Default Prompt JSON Parsing Failed")
            print("Raw Output:")
            print(output)
            return output, "", [], None, cls_reason, "", "classifier_fallback_failure" ### COMPELETE FAILURE ###

    ########## Stage 2 ###########
    output = "ERROR"

    for retry in range(max_retry):
        output = qwen_get_response(model, processor,image,template_v2(query, qtype),max_new_tokens=1024)

        try:
            description, targets, reasoning = parse_output_v2(output)
            return output, description, targets, qtype, cls_reason, reasoning, "complete_success"

        except:
            print(f"[ERROR] B: JSON Parsing Error - Retry {retry + 1}/{max_retry}")
            print("Raw Output:")
            print(output)

    #### FALLBACK 2 ###
    print("Specialized Prompt Failed - Use Default Prompt")
    output = qwen_get_response(model, processor,image,template_old(query)) ## FALLBACK ##

    try:
        description, targets = parse_output(output)
        return output, description, targets, qtype, cls_reason, "", "specialized_fallback_success"

    except:
        print("[ERROR] C: Default Prompt JSON Parsing Failed")
        print("Raw Output:")
        print(output)
        return output, "", [], qtype, cls_reason, "", "specialized_fallback_failure" ### COMPELETE FAILURE ###

## Functions for SAM

In [24]:
def expand_bbox(bbox, width, height, scale=1.05):
    x1, y1, x2, y2 = map(float, bbox)

    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    w = (x2 - x1) * scale
    h = (y2 - y1) * scale

    x1 = max(0, cx - w / 2)
    y1 = max(0, cy - h / 2)
    x2 = min(width, cx + w / 2)
    y2 = min(height, cy + h / 2)

    return [x1, y1, x2, y2]

def segment_targets_with_sam21(image, targets, bbox_scale=1.05):
    sam21_mask_predictor.set_image(np.array(image))

    final_mask = np.zeros((1000, 1000), dtype=bool)

    for target in targets:
        bbox = target["bbox"]
        point = target["point"]

        bbox = expand_bbox(
            bbox,
            width=image.width,
            height=image.height,
            scale=bbox_scale
        )

        input_box = np.array(bbox, dtype=np.float32)
        input_point = np.array([point])
        input_label = np.array([1])

        with torch.inference_mode():
            masks, scores, logits = sam21_mask_predictor.predict(
                point_coords=input_point,
                point_labels=input_label,
                box=input_box,
                multimask_output=True,
            )

        sorted_ind = np.argsort(scores)[::-1]
        masks = masks[sorted_ind]

        final_mask |= masks[0].astype(bool)

    return final_mask.astype(bool)

## Visualize Function

In [25]:
def show_bbox_point(image, targets):
    fig, ax = plt.subplots()

    ax.imshow(image)

    for target in targets:
        concept = target["concept"]
        bbox = target["bbox"]
        point = target["point"]

        x1, y1, x2, y2 = bbox
        x, y = point

        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            fill=False,
            edgecolor="red"
        )

        ax.add_patch(rect)
        ax.scatter(x, y, s=30, color="cyan")

    plt.show()

In [26]:
def show_mask(mask):
    plt.imshow(mask)
    plt.axis("off")
    plt.show()

### DELETE BEFORE SUBMIT

In [27]:
def mask_iou(mask1, mask2):
    mask1 = np.asarray(mask1).astype(bool)
    mask2 = np.asarray(mask2).astype(bool)

    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()

    if union == 0:
        return np.nan

    return intersection / union


def mask_dice(mask1, mask2):
    mask1 = np.asarray(mask1).astype(bool)
    mask2 = np.asarray(mask2).astype(bool)

    intersection = np.logical_and(mask1, mask2).sum()
    total = mask1.sum() + mask2.sum()

    if total == 0:
        return np.nan

    return 2 * intersection / total


def mask_area(mask):
    mask = np.asarray(mask).astype(bool)
    return int(mask.sum()), float(mask.mean())

In [28]:
def mask_to_bbox(mask):
    mask = np.asarray(mask).astype(bool)

    ys, xs = np.where(mask)

    if len(xs) == 0:
        return None

    x1 = int(xs.min())
    y1 = int(ys.min())
    x2 = int(xs.max()) + 1
    y2 = int(ys.max()) + 1

    return [x1, y1, x2, y2]

def bbox_iou(box1, box2):
    if box1 is None or box2 is None:
        return np.nan

    x1_a, y1_a, x2_a, y2_a = box1
    x1_b, y1_b, x2_b, y2_b = box2

    inter_x1 = max(x1_a, x1_b)
    inter_y1 = max(y1_a, y1_b)
    inter_x2 = min(x2_a, x2_b)
    inter_y2 = min(y2_a, y2_b)

    inter_area = (
        max(0, inter_x2 - inter_x1)
        * max(0, inter_y2 - inter_y1)
    )

    area_a = (x2_a - x1_a) * (y2_a - y1_a)
    area_b = (x2_b - x1_b) * (y2_b - y1_b)

    union_area = area_a + area_b - inter_area

    if union_area == 0:
        return np.nan

    return inter_area / union_area

In [29]:
def compare_results(
    mask_a,
    mask_b,
    targets_a=None,
    targets_b=None,
):
    area_a, area_ratio_a = mask_area(mask_a)
    area_b, area_ratio_b = mask_area(mask_b)

    result = {
        "mask_iou": mask_iou(mask_a, mask_b),
        "mask_dice": mask_dice(mask_a, mask_b),

        "mask_area_a": area_a,
        "mask_area_b": area_b,

        "mask_area_ratio_a": area_ratio_a,
        "mask_area_ratio_b": area_ratio_b,

        "area_ratio_difference": abs(area_ratio_a - area_ratio_b),

        "errors_a": check_errors(targets_a, mask_a)
        if targets_a is not None else [],

        "errors_b": check_errors(targets_b, mask_b)
        if targets_b is not None else [],
    }

    if targets_a:
        bbox_a, point_a = targets_a[0]
        result["bbox_mask_iou_a"] = bbox_iou(
            bbox_a,
            mask_to_bbox(mask_a)
        )
        result["point_inside_mask_a"] = point_in_mask(
            point_a,
            mask_a
        )

    if targets_b:
        bbox_b, point_b = targets_b[0]
        result["bbox_mask_iou_b"] = bbox_iou(
            bbox_b,
            mask_to_bbox(mask_b)
        )
        result["point_inside_mask_b"] = point_in_mask(
            point_b,
            mask_b
        )

    return result

## [Dev] Bbox and Point Error Check Function
Delete before Submition

In [30]:
def point_in_mask(point, mask):
    """
    Point가 MASK안에 있는지?를 true와 false로 반환하는 Function.
    """
    mask = np.asarray(mask).astype(bool)

    x, y = point
    x = int(round(x))
    y = int(round(y))
    h, w = mask.shape

    if not (0 <= x < w and 0 <= y < h):
        return False

    return bool(mask[y, x])

In [31]:
def point_in_bbox(point, bbox):
    """
    Point가 BBOX안에 있는지?를 true와 false로 반환하는 Function.
    """
    x, y = point
    x1, y1, x2, y2 = bbox

    return (x1 <= x < x2 and y1 <= y < y2)

In [32]:
def valid_bbox(bbox, image_shape=(1000, 1000)):
    h, w = image_shape
    x1, y1, x2, y2 = bbox
    return (0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h)

def valid_point(point, image_shape=(1000, 1000)):
    h, w = image_shape
    x, y = point

    return (0<= x<w and 0<= y<h)

In [33]:
def check_errors(targets, mask):
    errors = []

    mask = np.asarray(mask).astype(bool)

    for bbox, point in targets:
        if not valid_bbox(bbox):
            errors.append("invalid_bbox")

        if not valid_point(point):
            errors.append("invalid_point")

        if valid_bbox(bbox) and valid_point(point):
            if not point_in_bbox(point, bbox):
                errors.append("point_outside_bbox")

    if mask.shape != (1000, 1000):
        errors.append("shape_mismatch")

    if mask.sum() == 0:
        errors.append("empty_mask")

    if mask.sum() == mask.size:
        errors.append("full_mask")

    return errors

## <------- 공용 부분 종료 EOSS ------->
** 공용부분 필요시 확장. 수정시 PR 올리기

## Pipeline

In [34]:
masks=[] # Mask 결과 최종저장

# router 성능 측정변수 INIT
route_counts = Counter()
status_counts = Counter()
route_logs = []
masks = []

for i, (image_id, image, query) in enumerate(test_loader):
    # if i <10:
    #     continue
    # if i >= 20 :
    #     break
    # if not i in [4, 6, 11, 26, 35, 36, 59]:
    #     continue

    if QWEN3: ## ENV VARIABLE ##
        model=qwen3_model
        processor=qwen3_processor
    else:
        # model=qwen25_model
        # processor=qwen25_processor
        break 
    
    output, description, targets, qtype, cls_reason, reasoning, status = qwen_inference_v2(model,processor,image,query)
    route = qtype if qtype is not None else "DEFAULT"

    route_counts[route]+=1
    status_counts[status]+=1
    route_logs.append({
        "image_id": int(image_id),
        "query": query,
        "qtype": route,
        "status": status,
        "cls_reason": cls_reason,
        "num_targets": len(targets),
    })

    print(f"ID{image_id}: {query}")
    print(f"Query Type: {qtype} ({cls_reason})")
    print(f"Route Status: {status}")
    print(f"Reasoning: {reasoning}")
    print(f"Qwen Output: {output}")

    print("Description:", description)
    print("Targets:", targets)
    
    mask_sam21 = segment_targets_with_sam21(
        image,
        targets,
        bbox_scale=1.03
    )
    
    # mask_sam3 = segment_targets_with_sam3(
    #     image,
    #     targets,
    #     use_bbox=False
    # )

    # mask_sam3_bbox = segment_targets_with_sam3(
    #     image,
    #     targets,
    #     use_bbox=True
    # )

    show_bbox_point(image, targets)

    # print("SAM2.1")
    show_mask(mask_sam21)
    masks.append(mask_sam21)

    # print("SAM3 Concept Only")
    # show_mask(mask_sam3)

    # print("SAM3 Concept + BBox")
    # show_mask(mask_sam3_bbox)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ID0: The key contributor to the victory in the Wahlert vs. Hempstead matchup.
Query Type: A (The caption explicitly identifies Jim Hingtgen (12) as the senior quarterback who engineered Wahlert's victory by setting up one touchdown with a pass, throwing for another, and running for a third. This directly names the key contributor without requiring inference or additional reasoning.)
Route Status: complete_success
Reasoning: The caption identifies Jim Hingtgen (12) as the key contributor, stating he engineered the victory by setting up one touchdown, throwing for another, and running for a third. The image shows him in action.
Qwen Output: {
    "reasoning": "The caption identifies Jim Hingtgen (12) as the key contributor, stating he engineered the victory by setting up one touchdown, throwing for another, and running for a third. The image shows him in action.",
    "description": "Senior quarterback Jim Hingtgen (12) of Wahlert, the key contributor to the victory.",
    "targets": [
 

ID1: The person who is in the lead right now.
Query Type: D (The query asks to identify the person currently in the lead, which requires comparing the positions of multiple runners and selecting the one who is ahead of all others. This is a comparative task based on spatial positioning in the image.)
Route Status: complete_success
Reasoning: The runner in the lead is the one closest to the finish line, which is the runner in the innermost lane (lane 1) with the most forward position. This runner is wearing a blue and yellow uniform with 'GIBSON' on the bib.
Qwen Output: {
    "reasoning": "The runner in the lead is the one closest to the finish line, which is the runner in the innermost lane (lane 1) with the most forward position. This runner is wearing a blue and yellow uniform with 'GIBSON' on the bib.",
    "description": "The lead runner in the race, wearing a blue and yellow uniform with 'GIBSON' on the bib.",
    "targets": [
        {
            "concept": "lead runner",
     

ID2: The silent guardians that stand at the threshold, their presence alone enough to keep evil at bay.
Query Type: B (The query uses metaphorical language ('silent guardians', 'keep evil at bay') to describe the stone lion statues (komainu) flanking the shrine entrance, which are culturally recognized as protective deities in Shinto tradition. This requires interpreting implied visual and cultural clues rather than direct naming.)
Route Status: complete_success
Reasoning: The query metaphorically refers to guardian statues, typically lion-dogs (komainu), positioned at shrine entrances to ward off evil. The image shows two such stone statues flanking the main gate of a Japanese shrine, matching the description perfectly.
Qwen Output: {
    "reasoning": "The query metaphorically refers to guardian statues, typically lion-dogs (komainu), positioned at shrine entrances to ward off evil. The image shows two such stone statues flanking the main gate of a Japanese shrine, matching the descri

ID3: In the current Sudoku puzzle, among the bold-lined 3×3 boxes, locate the cell in the box with the most pre-filled numbers where 4 is the only possible value.
Query Type: E (The query requires solving a Sudoku puzzle by analyzing the pre-filled numbers in each 3x3 box, identifying the box with the most pre-filled numbers, and then determining which cell in that box can only contain the number 4. This involves logical deduction and step-by-step reasoning to eliminate possibilities, which falls under quantitative/logical reasoning.)
Route Status: complete_success
Reasoning: The top-right 3×3 box has 5 pre-filled numbers (2, 6, 8, 5, 4), which is the highest among all boxes. Within this box, the cell at row 3, column 9 (bottom-right corner) is empty and can only be 4, as all other numbers 1-9 are already present in its row, column, or box.
Qwen Output: {
    "reasoning": "The top-right 3×3 box has 5 pre-filled numbers (2, 6, 8, 5, 4), which is the highest among all boxes. Within this 

ID4: Which button would you press if your car had to stop unexpectedly to warn approaching drivers?
Query Type: C (The query describes a situational need (warning approaching drivers during an unexpected stop), which requires common-sense knowledge of what car function serves that purpose. The correct answer is typically the hazard lights button, which is not explicitly named but can be inferred from the context of the situation and general car functionality.)
Route Status: complete_success
Reasoning: The query asks for a button to warn approaching drivers in an emergency stop. The hazard warning button, typically marked with a triangle icon, is the standard control for this purpose. In the image, the red triangle button is located on the right side of the central console, near the climate controls, and is clearly labeled with a hazard symbol.
Qwen Output: {
    "reasoning": "The query asks for a button to warn approaching drivers in an emergency stop. The hazard warning button, typica

... (191 outputs left out)

ID196: The surface you would use to combine different paints into a new shade.
Query Type: C (The query describes a functional need — combining paints to create new shades — which requires identifying an object based on its common use in art. This is a situational commonsense reasoning task, as the correct object (a paint palette) is inferred from understanding typical artistic practices.)
Route Status: complete_success
Reasoning: The query asks for a surface used to mix paints. The image shows a watercolor palette with a built-in mixing area, which is the standard tool for this purpose. The mixing area is the functional component that satisfies the condition.
Qwen Output: {
    "reasoning": "The query asks for a surface used to mix paints. The image shows a watercolor palette with a built-in mixing area, which is the standard tool for this purpose. The mixing area is the functional component that satisfies the condition.",
    "description": "The mixing area of the watercolor palette"

ID197: In the current scene, find all people whose reflections are not visible in the mirror.
Query Type: A (The query asks to identify people whose reflections are not visible in the mirror. This requires observing which individuals are positioned such that their reflections are not captured in the mirror's surface, which is a direct visual identification task based on spatial positioning relative to the mirror. No inference, comparison, or calculation is needed beyond direct observation.)
Route Status: complete_success
Reasoning: The children in the foreground are not reflected in the mirror because they are positioned in front of it, while the mirror reflects only those behind them. The girl in the purple shirt is also not reflected as she is partially out of the mirror's field of view.
Qwen Output: {
    "reasoning": "The children in the foreground are not reflected in the mirror because they are positioned in front of it, while the mirror reflects only those behind them. The girl 

ID198: Among the countries shown on this map, segment the country that shares borders with the greatest number of other countries.
Query Type: D (The query requires comparing the number of neighboring countries for each country shown on the map and identifying the one with the greatest count. This involves examining the map's borders and performing a comparative analysis, which falls under comparative reasoning.)
Route Status: complete_success
Reasoning: Thailand is surrounded by five countries: Myanmar, Laos, Cambodia, Vietnam, and Malaysia. This is the highest number of shared borders among all countries shown on the map.
Qwen Output: {
    "reasoning": "Thailand is surrounded by five countries: Myanmar, Laos, Cambodia, Vietnam, and Malaysia. This is the highest number of shared borders among all countries shown on the map.",
    "description": "The country with the most neighboring countries on the map.",
    "targets": [
        {
            "concept": "Thailand",
            "bbo

ID199: Among the dice on which two faces are visible at the same time, mark the die whose visible faces have the larger sum.
Query Type: D (The query requires comparing the sums of visible faces on dice where two faces are visible, which involves identifying such dice and calculating their face sums to determine which has the larger sum. This is a comparative task based on visual inspection and arithmetic, fitting the 'Comparative / Ranking' reasoning type.)
Route Status: complete_success
Reasoning: Among the stacked dice, only the top die shows two faces (top and front). The top die shows 1 and 3, summing to 4. The bottom die shows 1 and 3, also summing to 4. Since both have the same sum, the top die is selected as it is the first one meeting the condition.
Qwen Output: {
    "reasoning": "Among the stacked dice, only the top die shows two faces (top and front). The top die shows 1 and 3, summing to 4. The bottom die shows 1 and 3, also summing to 4. Since both have the same sum, the 

## Save Submission

In [35]:
!pwd

/home/bootcamp2026_32


In [36]:
submission_path = "sample_submission2.csv"

submission = pd.read_csv(submission_path)

rles = []

for mask in masks:
    mask = np.asarray(mask).squeeze()
    mask = mask.astype(bool)

    rle = encode_single_mask(mask)   # 제공된 RLE 함수
    rles.append(rle)

submission["label"] = rles

submission.to_csv("submission.csv", index=False)

In [37]:
for i, mask in enumerate(masks):
    ratio = mask.mean()

    if ratio < 0.001 or ratio > 0.95:
        print(i, ratio)

In [38]:
print(submission.shape)
print(submission["label"].isna().sum())
print(submission["ID"].iloc[0])
print(submission["ID"].iloc[-1])

(200, 2)
0
test_00000
test_00199


In [39]:
print("\n=== Route Counts ===")
for key in ["A", "B", "C", "D", "E", "F", "DEFAULT"]:
    print(f"{key}: {route_counts[key]}")

print("\n=== Route Status ===")
for key in [
    "complete_success",
    "classifier_fallback_success",
    "classifier_fallback_failure",
    "specialized_fallback_success",
    "specialized_fallback_failure",
]:
    print(f"{key}: {status_counts[key]}")


=== Route Counts ===
A: 33
B: 27
C: 77
D: 24
E: 11
F: 28
DEFAULT: 0

=== Route Status ===
complete_success: 199
classifier_fallback_success: 0
classifier_fallback_failure: 0
specialized_fallback_success: 1
specialized_fallback_failure: 0


In [40]:
route_df = pd.DataFrame(route_logs)
route_df.to_csv("route_logs.csv", index=False)